# Generate 2023 UK Data Sythetically

### Vehicle Registrations Trends UK 2024 vs 2023

https://www.acea.auto/pc-registrations/new-car-registrations-0-8-in-2024-battery-electric-13-6-market-share/

In [107]:
total_count_2023 = [131994, 74441, 287825, 132990, 145204, 177266, 86657, 272610, 153529, 156525, 141092]
total_count_2024 = [147876, 84886, 317768, 134274, 147678, 179263, 84575, 275239, 144288, 153610, 140786]

# calculate monthly percentage change
monthly_percentage_change = [round((old - new) / new * 100, 4) for old, new in zip(total_count_2023, total_count_2024)]
trend_adjustment_map = dict(zip(range(1, 12), monthly_percentage_change))
trend_adjustment_map

{1: -10.7401,
 2: -12.3047,
 3: -9.4229,
 4: -0.9563,
 5: -1.6753,
 6: -1.114,
 7: 2.4617,
 8: -0.9552,
 9: 6.4046,
 10: 1.8977,
 11: 0.2174}

In [108]:
import pandas as pd
df = pd.read_parquet("../data/fact_registered_vehicles.parquet")

In [109]:
id_columns = ['oem', 'year_report', 'month_report', 'level_0_country',
       'level_1_region_name', 'level_2_district_postcode',
       'level_2_district_town_name', 
       'model_class', 'model_class_specific', 'body_type', 'fuel_type',
       'doors', 'Energy_Source']

# id_columns_year_id = ['oem', 'year_report', 'level_0_country',
#        'level_1_region_name', 'level_2_district_postcode',
#        'level_2_district_town_name', 
#        'model_class', 'model_class_specific', 'body_type', 'fuel_type',
#        'doors', 'Energy_Source']

df['vehicle_count_id'] = df[id_columns].apply(lambda x: '_'.join(x.astype(str)), axis=1)
#df['vehicle_count_id_year'] = df[id_columns_year_id].apply(lambda x: '_'.join(x.astype(str)), axis=1)

## Generate 2023 Data using Trend Adjustment Map

In [110]:
df_2023 = df[['vehicle_count_id','month_report', 'vehicle_count']].copy()
df_2023['vehicle_count_id'] = df_2023['vehicle_count_id'].str.replace('_2024_', '_2023_')
df_2023['year_report'] = 2023

# Apply Trend Adjustment Map if and only if vehicle count > 1
import numpy as np
df_2023['vehicle_count'] = np.where(df_2023['vehicle_count'] > 1, np.floor(df_2023['vehicle_count'] * (1 + df_2023['month_report'].map(trend_adjustment_map).fillna(0) / 100)), df_2023['vehicle_count'])
df_2023['vehicle_count'] = df_2023['vehicle_count'].astype(int)

In [111]:
df_2023.head()

,vehicle_count_id,month_report,vehicle_count,year_report
0,ASTON MARTIN_2023_10_England_Bedford Borough_M...,10,1,2023
1,ASTON MARTIN_2023_10_England_Bedford Borough_M...,10,1,2023
2,ASTON MARTIN_2023_10_England_Bedford Borough_M...,10,1,2023
3,ASTON MARTIN_2023_10_England_Bracknell Forest_...,10,1,2023
4,ASTON MARTIN_2023_10_England_Bradford_LS29_Ilk...,10,1,2023


In [112]:
assert df_2023.shape[0] == df.shape[0], "Data frame shapes do not match"

In [113]:
monthly_totals_2024 = df.groupby(['month_report']).agg({'vehicle_count': 'sum'}).reset_index().set_index('month_report').rename(columns={'vehicle_count': 'total_vehicle_count_2024'})
monthly_totals_2023 = df_2023.groupby(['month_report']).agg({'vehicle_count': 'sum'}).reset_index().set_index('month_report').rename(columns={'vehicle_count': 'total_vehicle_count_2023'})

# calculate monthly percentage change 2024 to 2023
monthly_percentage_change = ((monthly_totals_2023['total_vehicle_count_2023'] - monthly_totals_2024['total_vehicle_count_2024']) / monthly_totals_2024['total_vehicle_count_2024'] * 100).fillna(0)
monthly_percentage_change

month_report
1    -10.415525
2    -10.287299
3    -10.859718
4     -7.761241
5     -8.365366
6     -8.845687
7      0.241094
8     -7.851181
9      1.280483
10     0.080785
11     0.000000
12     0.000000
dtype: float64

In [114]:
trend_adjustment_map

{1: -10.7401,
 2: -12.3047,
 3: -9.4229,
 4: -0.9563,
 5: -1.6753,
 6: -1.114,
 7: 2.4617,
 8: -0.9552,
 9: 6.4046,
 10: 1.8977,
 11: 0.2174}

In [115]:
df[df['vehicle_count_id'] == 'AUDI_2024_3_England_Leicester_LE1_Leicester_Q2_TFSI S LINE_SUV_PETROL_5_NORMAL']

,vehicle_count_id,vehicle_count,oem,year_report,month_report,level_0_country,level_1_region_name,level_2_district_postcode,level_2_district_town_name,model_class,model_class_specific,body_type,fuel_type,doors,Energy_Source
29198,AUDI_2024_3_England_Leicester_LE1_Leicester_Q2...,42,AUDI,2024,3,England,Leicester,LE1,Leicester,Q2,TFSI S LINE,SUV,PETROL,5,NORMAL


In [116]:
df_2023[df_2023['vehicle_count_id'] == 'AUDI_2023_3_England_Leicester_LE1_Leicester_Q2_TFSI S LINE_SUV_PETROL_5_NORMAL']

,vehicle_count_id,month_report,vehicle_count,year_report
29198,AUDI_2023_3_England_Leicester_LE1_Leicester_Q2...,3,38,2023


In [133]:
id_columns = ['oem', 'year_report', 'month_report', 'level_0_country',
       'level_1_region_name', 'level_2_district_postcode',
       'level_2_district_town_name', 
       'model_class', 'model_class_specific', 'body_type', 'fuel_type',
       'doors', 'Energy_Source']

# extract columns from vehicle_count_id 
df_2023['oem'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[0]
df_2023['year_report'] = (df_2023['vehicle_count_id'].str.split('_', expand=True)[1]).astype(int)
df_2023['month_report'] = (df_2023['vehicle_count_id'].str.split('_', expand=True)[2]).astype(int)
df_2023['level_0_country'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[3]
df_2023['level_1_region_name'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[4]
df_2023['level_2_district_postcode'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[5]
df_2023['level_2_district_town_name'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[6]
df_2023['model_class'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[7]
df_2023['model_class_specific'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[8]
df_2023['body_type'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[9]
df_2023['fuel_type'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[10]
df_2023['doors'] = (df_2023['vehicle_count_id'].str.split('_', expand=True)[11]).astype(int)
df_2023['Energy_Source'] = df_2023['vehicle_count_id'].str.split('_', expand=True)[12]

In [134]:
df_2023.filter(['vehicle_count_id'] + id_columns + ['vehicle_count']).head()

,vehicle_count_id,oem,year_report,month_report,level_0_country,level_1_region_name,level_2_district_postcode,level_2_district_town_name,model_class,model_class_specific,body_type,fuel_type,doors,Energy_Source,vehicle_count
0,ASTON MARTIN_2023_10_England_Bedford Borough_M...,ASTON MARTIN,2023,10,England,Bedford Borough,MK43,Kempston Rural,DBX,V8,SUV,PETROL,5,NORMAL,1
1,ASTON MARTIN_2023_10_England_Bedford Borough_M...,ASTON MARTIN,2023,10,England,Bedford Borough,MK44,Wilden,DBX,V8,SUV,PETROL,5,NORMAL,1
2,ASTON MARTIN_2023_10_England_Bedford Borough_M...,ASTON MARTIN,2023,10,England,Bedford Borough,MK44,Wilden,VANTAGE,V8,SPORTSCOUPE,PETROL,3,NORMAL,1
3,ASTON MARTIN_2023_10_England_Bracknell Forest_...,ASTON MARTIN,2023,10,England,Bracknell Forest,RG42,Warfield,DBX,V8,SUV,PETROL,5,NORMAL,1
4,ASTON MARTIN_2023_10_England_Bradford_LS29_Ilk...,ASTON MARTIN,2023,10,England,Bradford,LS29,Ilkley,VANTAGE,V8,SPORTSCOUPE,PETROL,3,NORMAL,1


In [135]:
assert set(df_2023.shape) == set(df.shape), "Data frame shapes do not match"

In [136]:
df_new = pd.concat([df, df_2023], ignore_index=True).reset_index(drop=True)

In [ ]:
assert set(df_new['year_report'].unique().tolist())==set([2023, 2024]), "Expected years in the data 2023 and 2024 are not found"

True

In [139]:
df_new.shape

(637590, 15)

In [140]:
df_new.head()

,vehicle_count_id,vehicle_count,oem,year_report,month_report,level_0_country,level_1_region_name,level_2_district_postcode,level_2_district_town_name,model_class,model_class_specific,body_type,fuel_type,doors,Energy_Source
0,ASTON MARTIN_2024_10_England_Bedford Borough_M...,1,ASTON MARTIN,2024,10,England,Bedford Borough,MK43,Kempston Rural,DBX,V8,SUV,PETROL,5,NORMAL
1,ASTON MARTIN_2024_10_England_Bedford Borough_M...,1,ASTON MARTIN,2024,10,England,Bedford Borough,MK44,Wilden,DBX,V8,SUV,PETROL,5,NORMAL
2,ASTON MARTIN_2024_10_England_Bedford Borough_M...,1,ASTON MARTIN,2024,10,England,Bedford Borough,MK44,Wilden,VANTAGE,V8,SPORTSCOUPE,PETROL,3,NORMAL
3,ASTON MARTIN_2024_10_England_Bracknell Forest_...,1,ASTON MARTIN,2024,10,England,Bracknell Forest,RG42,Warfield,DBX,V8,SUV,PETROL,5,NORMAL
4,ASTON MARTIN_2024_10_England_Bradford_LS29_Ilk...,1,ASTON MARTIN,2024,10,England,Bradford,LS29,Ilkley,VANTAGE,V8,SPORTSCOUPE,PETROL,3,NORMAL


In [141]:
df_new.to_parquet("../data/fact_registered_vehicles_2023_2024.parquet", index=False)